# Model A complete analysis: H1, PPC, M1 and memory-bounded PSIS-LOO

This standalone Colab notebook completes only Model A. It reuses the successful full-design base
fit and never samples it again. It then calculates H1, runs the Model A recovery and
posterior-predictive gates, fits the three missing M1 variants, runs the required sensitivity
checks, and computes the four PSIS-LOO results in bounded and resumable blocks.

No Chronos inference or data collection is repeated. Every expensive stage is checkpointed.
After a Colab disconnect, use Runtime > Run all and completed work is reused.

Terminology:

- posterior: parameter distributions after observing the data;
- PPC: posterior predictive check, comparing model replications with observed data;
- PSIS-LOO: efficient leave-one-out predictive model comparison;
- R-hat, ESS and divergences: chain agreement, effective independent samples and sampler warnings.

Expected remaining work is three M1 fits, three sensitivity fits, one reduced synthetic recovery
fit, one blocked PPC and four blocked LOO calculations. On a runtime matching the 12.85-minute
base fit, allow roughly 2 to 3 hours in total.


## 1. Setup

### 1.1 Locate or clone the repository

The notebook reuses the project checkpoint, diagnostics and design helpers.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IS_COLAB = on_colab()


def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target


REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))

print("repository:", REPO)
print("modules   :", BAYES_DIR)
print("runtime   :", "Colab" if IS_COLAB else "local")


### 1.2 Install the locked environment

The first run can deliberately restart Colab. When it reconnects, choose Runtime > Run all again.


In [ ]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A_pilot_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf", "h5py"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this pilot.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            subprocess.check_call(
                [
                    UV,
                    "pip",
                    "install",
                    "--python",
                    sys.executable,
                    "--reinstall-package",
                    "numpy",
                    "--reinstall-package",
                    "scipy",
                ]
            )
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


### 1.3 Analysis settings and Drive paths

The completed base fit remains read-only. All new artifacts use a separate versioned folder.


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import inspect
import json
import math
import os
import random
import time
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr
from IPython.display import display
from scipy import stats
from scipy.special import gammaln, logsumexp

import bayesian_checks as bc
import checkpointing as cp
import probe_lib as pl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

NOTEBOOK_VERSION = "model-A-complete-v1"
MODEL_VERSION = "A-centred-noncentered-zerosum-bg-v2"
SOURCE_RUN_ID = "d3_15model_v1"
BASE_RUN_ID = "model_A_full_noncentered_zerosum_bg_v1"
OUTPUT_RUN_ID = "model_A_complete_noncentered_zerosum_bg_v1"

DRAWS, TUNE, CHAINS, CORES = 2000, 2000, 4, 2
TARGET_ACCEPT = 0.90
NUTS_BACKEND = "nutpie"
PRIOR_SCALE, PRIOR_LADDER, NU = 0.5, (0.25, 0.5, 1.0), 4
ATTENUATION_20 = float(np.log(0.8))
ROPE_LOG = float(np.log(1.1))
PPC_DRAW_CHUNK = 25
LOO_TARGET_BYTES = 128 * 1024**2
PPC_MIN_COVERAGE = 0.90
SENSITIVITY_MAX_SPREAD = 0.10

# Required by the current project gate. Turning one off produces NOT REPORTABLE, not a shortcut.
RUN_PARAMETER_RECOVERY = True
RUN_SENSITIVITY = True

if NUTS_BACKEND == "nutpie" and importlib.util.find_spec("nutpie") is None:
    print("nutpie unavailable; falling back to PyMC")
    NUTS_BACKEND = "pymc"

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/patchAliasing")
except (ImportError, ModuleNotFoundError):
    DRIVE_ROOT = BAYES_DIR / "_run"

SOURCE_ROOT = DRIVE_ROOT / "full" / SOURCE_RUN_ID
SOURCE_FILE = SOURCE_ROOT / "data" / "02_contrasts.parquet"
BASE_ROOT = DRIVE_ROOT / "pilots" / BASE_RUN_ID
BASE_CHECKPOINT = BASE_ROOT / "04_A_zerosum_pilot.nc"
BASE_MANIFEST_PATH = BASE_ROOT / "pilot_manifest.json"
OUTPUT_ROOT = DRIVE_ROOT / "full" / OUTPUT_RUN_ID
FIGURE_ROOT = OUTPUT_ROOT / "figures"
WORK_ROOT = OUTPUT_ROOT / "_work"
for directory in (OUTPUT_ROOT, FIGURE_ROOT, WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

style = next((x for x in ("arviz-whitegrid", "seaborn-v0_8-whitegrid")
              if x in plt.style.available), "default")
plt.style.use(style)
print("notebook/model :", NOTEBOOK_VERSION, MODEL_VERSION)
print("source         :", SOURCE_FILE)
print("base fit       :", BASE_CHECKPOINT)
print("new outputs    :", OUTPUT_ROOT)
print("draws/tune/chains/cores:", DRAWS, TUNE, CHAINS, CORES)
print("sampler        :", NUTS_BACKEND)
print("blocks         : PPC", PPC_DRAW_CHUNK, "draws; LOO", LOO_TARGET_BYTES // 2**20, "MiB")


## 2. Data and model

### 2.1 Load and validate the frozen full-design contrasts

The notebook requires all 15 configurations and background IDs 0 through 99 for both generators.


In [ ]:
REQUIRED_COLUMNS = {
    "model", "P", "S", "overlap", "generator", "bg_id", "f_lock", "phase_idx", "d", "live"
}
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"missing completed contrast table: {SOURCE_FILE}")
contrasts = pd.read_parquet(SOURCE_FILE, columns=sorted(REQUIRED_COLUMNS))
missing = REQUIRED_COLUMNS - set(contrasts.columns)
if missing:
    raise ValueError(f"contrast table is missing columns: {sorted(missing)}")

expected_models = {pl.model_tag(P, S) for P, S in pl.DELIVERABLE3_MODELS}
expected_generators = set(pl.GENERATORS)
if set(contrasts["model"].astype(str)) != expected_models:
    raise ValueError("source does not contain the frozen 15-model design")
if set(contrasts["generator"].astype(str)) != expected_generators:
    raise ValueError("source generator coverage differs from the frozen design")
if contrasts[["model", "generator", "bg_id", "f_lock", "phase_idx"]].duplicated().any():
    raise ValueError("duplicate Model A contrast keys")
if not np.isfinite(contrasts["d"].to_numpy(float)).all():
    raise ValueError("contrast d contains NaN or infinite values")

background_ids = {
    generator: sorted(map(int, contrasts.loc[contrasts["generator"].eq(generator), "bg_id"].unique()))
    for generator in sorted(expected_generators)
}
if any(ids != list(range(100)) for ids in background_ids.values()):
    raise ValueError("expected background IDs 0 through 99 for both generators")

live = (
    contrasts[contrasts["live"].astype(bool)]
    .sort_values(["model", "generator", "bg_id", "f_lock", "phase_idx"])
    .reset_index(drop=True)
)
coverage = live.groupby(["model", "generator"], observed=True).size().rename("n_live").reset_index()
if len(coverage) != len(expected_models) * len(expected_generators):
    raise ValueError("a model by generator live cell is missing")
SOURCE_SHA256 = cp.sha256_file(SOURCE_FILE)
print("source rows       :", len(contrasts))
print("full live rows    :", len(live))
print("source sha256     :", SOURCE_SHA256)
display(coverage.pivot(index="model", columns="generator", values="n_live"))
del contrasts
gc.collect()


### 2.2 Define the successful Model A family

The four M1 models differ only in the configuration-level overlap and patch-size predictors.
Background effects use the non-centred, sum-to-zero form that passed the full run.


In [ ]:
def codes(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    integer_codes, levels = pd.factorize(series, sort=False)
    return np.asarray(integer_codes, dtype=int), np.asarray(levels)


def overlap_scaled(df: pd.DataFrame, levels: np.ndarray) -> np.ndarray:
    value = df.groupby("model", observed=True)["overlap"].first().reindex(levels).to_numpy(float)
    return (value - value.mean()) / 0.5


def log_patch_centred(df: pd.DataFrame, levels: np.ndarray) -> np.ndarray:
    value = df.groupby("model", observed=True)["P"].first().reindex(levels).to_numpy(float)
    value = np.log(value)
    return value - value.mean()


def model_A(df: pd.DataFrame, scale=PRIOR_SCALE, nu=NU,
            likelihood="student", config_level="both") -> pm.Model:
    if config_level not in {"both", "overlap", "patch", "none"}:
        raise ValueError(f"unknown config level: {config_level}")
    if likelihood not in {"student", "normal"}:
        raise ValueError(f"unknown likelihood: {likelihood}")
    config_code, config_levels = codes(df["model"])
    harmonic_code, harmonic_levels = codes(df["f_lock"].round(3).astype(str))
    background_code, background_levels = codes(
        df["generator"].astype(str) + "#" + df["bg_id"].astype(str)
    )
    overlap = overlap_scaled(df, config_levels)
    log_patch = log_patch_centred(df, config_levels)
    observed = df["d"].to_numpy(float)
    coords = {
        "config": config_levels, "harmonic": harmonic_levels,
        "background": background_levels, "obs": np.arange(len(observed)),
    }
    with pm.Model(coords=coords) as model:
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=scale)
        config_mean = beta_bar
        if config_level in {"both", "overlap"}:
            delta_O = pm.StudentT("delta_O", nu=nu, mu=0.0, sigma=scale)
            config_mean = config_mean + delta_O * overlap
        if config_level in {"both", "patch"}:
            delta_P = pm.StudentT("delta_P", nu=nu, mu=0.0, sigma=scale)
            config_mean = config_mean + delta_P * log_patch
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Normal("beta", mu=config_mean, sigma=tau, dims="config")
        sigma_harm = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_harm = pm.ZeroSumNormal("u_harm", sigma=sigma_harm, dims="harmonic")
        sigma_bg = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        z_bg = pm.ZeroSumNormal("z_bg", sigma=1.0, dims="background")
        u_bg = pm.Deterministic("u_bg", sigma_bg * z_bg, dims="background")
        location = beta[config_code] + u_harm[harmonic_code] + u_bg[background_code]
        sigma = pm.HalfStudentT("sigma", nu=nu, sigma=scale)
        if likelihood == "normal":
            pm.Normal("d", mu=location, sigma=sigma, observed=observed, dims="obs")
        else:
            pm.StudentT("d", nu=nu, mu=location, sigma=sigma, observed=observed, dims="obs")
        pm.Deterministic("recovery_ratio", pm.math.exp(beta_bar))
    return model


structure = model_A(live)
print("model version:", MODEL_VERSION)
print("free variables:", [variable.name for variable in structure.free_RVs])
print({name: len(value) for name, value in structure.coords.items()})
del structure
gc.collect()


### 2.3 Verify the base checkpoint and create the new immutable manifest

The base is accepted only when its data hash, model version, full sampling settings and artifact
hash match. New outputs are hashed in a separate manifest.


In [ ]:
if not BASE_MANIFEST_PATH.is_file() or not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError("the completed full A checkpoint or manifest is missing")
base_manifest = json.loads(BASE_MANIFEST_PATH.read_text(encoding="utf-8"))
base_spec = base_manifest.get("pilot_spec", {})
base_sampling = base_spec.get("sampling", {})
for key, expected in {
    "model_version": MODEL_VERSION,
    "source_sha256": SOURCE_SHA256,
    "selected_live_rows": len(live),
}.items():
    if base_spec.get(key) != expected:
        raise ValueError(f"base mismatch for {key}: {base_spec.get(key)!r} != {expected!r}")
for key, expected in {
    "draws": DRAWS, "tune": TUNE, "chains": CHAINS, "log_likelihood": False
}.items():
    if base_sampling.get(key) != expected:
        raise ValueError(f"base sampling mismatch for {key}")
if any(len(ids) != 100 for ids in base_spec.get("selected_bg_ids", {}).values()):
    raise ValueError("base fit did not use 100 backgrounds per generator")
base_entry = base_manifest.get("artifacts", {}).get(BASE_CHECKPOINT.name)
if not base_entry:
    raise ValueError("base checkpoint is not recorded in its manifest")
BASE_SHA256 = cp.sha256_file(BASE_CHECKPOINT)
if BASE_SHA256 != base_entry.get("sha256"):
    raise ValueError("base checkpoint hash differs from its manifest")
base_result = base_manifest.get("pilot_result", {})
if not all(base_result.get(key) is True for key in
           ("pilot_global_ok", "headline_ok", "combined_location_ok")):
    raise ValueError("base full A convergence result is not a complete PASS")

MANIFEST_PATH = OUTPUT_ROOT / "analysis_manifest.json"
ANALYSIS_SPEC = {
    "notebook_version": NOTEBOOK_VERSION,
    "model_version": MODEL_VERSION,
    "source_sha256": SOURCE_SHA256,
    "base_checkpoint_sha256": BASE_SHA256,
    "rows": len(live),
    "sampling": {
        "draws": DRAWS, "tune": TUNE, "chains": CHAINS, "cores": CORES,
        "target_accept": TARGET_ACCEPT, "backend": NUTS_BACKEND, "log_likelihood": False,
    },
    "thresholds": {
        "rhat_max": bc.RHAT_MAX, "ess_min": bc.ESS_MIN,
        "posterior_cutoff": bc.POSTERIOR_CUTOFF,
        "ppc_min_coverage": PPC_MIN_COVERAGE,
        "sensitivity_max_spread": SENSITIVITY_MAX_SPREAD,
    },
    "blocks": {"ppc_draw_chunk": PPC_DRAW_CHUNK, "loo_target_bytes": LOO_TARGET_BYTES},
    "helper_hashes": {
        name: cp.sha256_file(BAYES_DIR / name)
        for name in ("bayesian_checks.py", "checkpointing.py", "probe_lib.py")
    },
}
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
if MANIFEST_PATH.is_file():
    analysis_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if analysis_manifest.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("output manifest mismatch; increment OUTPUT_RUN_ID")
else:
    analysis_manifest = {
        "schema_version": 1, "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "analysis_spec": ANALYSIS_SPEC, "artifacts": {},
    }
cp.atomic_json(MANIFEST_PATH, analysis_manifest)


def artifact_path(name: str) -> Path:
    return OUTPUT_ROOT / name


def record_artifact(name: str) -> None:
    path = artifact_path(name)
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if current.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("analysis manifest changed during this runtime")
    analysis_manifest["artifacts"] = current.get("artifacts", {})
    analysis_manifest["artifacts"][name] = {
        "sha256": cp.sha256_file(path), "bytes": path.stat().st_size
    }
    cp.atomic_json(MANIFEST_PATH, analysis_manifest)


def have_artifact(name: str) -> bool:
    path = artifact_path(name)
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    entry = current.get("artifacts", {}).get(name)
    if path.exists() != (entry is not None):
        raise ValueError(f"untracked or missing artifact: {name}")
    if not path.exists():
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"artifact hash mismatch: {name}")
    return True


def save_idata(idata: az.InferenceData, name: str) -> None:
    cp.atomic_netcdf(artifact_path(name), idata)
    record_artifact(name)
    print("checkpoint saved:", artifact_path(name))


def load_idata(path: Path) -> az.InferenceData:
    return az.from_netcdf(path)


def save_frame(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    cp.atomic_parquet(artifact_path(name), frame)
    record_artifact(name)
    return frame


def load_frame(name: str) -> pd.DataFrame:
    if not have_artifact(name):
        raise FileNotFoundError(name)
    return pd.read_parquet(artifact_path(name))


def save_value(value, name: str):
    cp.atomic_json(artifact_path(name), value)
    record_artifact(name)
    return value


def load_value(name: str):
    if not have_artifact(name):
        raise FileNotFoundError(name)
    return json.loads(artifact_path(name).read_text(encoding="utf-8"))


print("base fit verified:", BASE_SHA256)
print("base fit minutes :", round(float(base_result.get("fit_seconds", 0)) / 60, 2))
print("output manifest  :", MANIFEST_PATH)


## 3. Reuse the full A posterior and calculate H1

No fitting occurs here. H1 support is at least 20 percent attenuation with posterior probability
at least 0.95. Refutation requires at least 0.95 probability inside the project 10 percent
practical-equivalence region. The final verdict waits for all validation gates.


In [ ]:
idata_A = load_idata(BASE_CHECKPOINT)
if int(idata_A.posterior.sizes["chain"]) != CHAINS:
    raise ValueError("base posterior chain count mismatch")
if int(idata_A.posterior.sizes["draw"]) != DRAWS:
    raise ValueError("base posterior draw count mismatch")


def posterior_report(idata, variable, transform=None, label=None):
    values = np.asarray(idata.posterior[variable], dtype=float).ravel()
    if transform is not None:
        values = transform(values)
    low, high = np.quantile(values, [0.025, 0.975])
    return {
        "parameter": label or variable, "median": float(np.median(values)),
        "hdi_low": float(low), "hdi_high": float(high), "sd": float(np.std(values)),
    }


def posterior_probability(idata, variable, predicate):
    values = np.asarray(idata.posterior[variable], dtype=float).ravel()
    return float(np.mean(predicate(values)))


h1_summary = pd.DataFrame([
    posterior_report(idata_A, "beta_bar", label="beta_bar (log recovery ratio)"),
    posterior_report(idata_A, "beta_bar", transform=np.exp,
                     label="exp(beta_bar) recovery ratio"),
    posterior_report(idata_A, "delta_O", label="delta_O (overlap mitigation)"),
    posterior_report(idata_A, "delta_P", label="delta_P (log patch-size slope)"),
])

P_ATTENUATION_20 = posterior_probability(
    idata_A, "beta_bar", lambda x: x < ATTENUATION_20
)
P_BETA_ROPE = posterior_probability(
    idata_A, "beta_bar", lambda x: np.abs(x) < ROPE_LOG
)
P_BETA_NEGATIVE = posterior_probability(idata_A, "beta_bar", lambda x: x < 0)
P_DELTA_O_POSITIVE = posterior_probability(idata_A, "delta_O", lambda x: x > 0)
base_diagnostic = bc.fit_diagnostics("A", idata_A, az)

save_frame(h1_summary, "04_A_h1_summary_v1.parquet")
save_value({
    "P_attenuation_20": P_ATTENUATION_20,
    "P_beta_in_rope": P_BETA_ROPE,
    "P_beta_negative": P_BETA_NEGATIVE,
    "P_delta_O_positive": P_DELTA_O_POSITIVE,
    "diagnostics": base_diagnostic,
}, "04_A_h1_probabilities_v1.json")

display(h1_summary.round(5))
display(pd.DataFrame([base_diagnostic]).round(5))
print(f"P(beta_bar < log 0.8 | D) = {P_ATTENUATION_20:.4f}")
print(f"P(abs(beta_bar) < log 1.1 | D) = {P_BETA_ROPE:.4f}")
print(f"P(beta_bar < 0 | D) = {P_BETA_NEGATIVE:.4f}")
print(f"P(delta_O > 0 | D) = {P_DELTA_O_POSITIVE:.4f}")

beta = idata_A.posterior["beta"]
levels = list(map(str, beta.coords["config"].values))
hdi = np.asarray(az.hdi(beta, hdi_prob=0.95)["beta"], dtype=float)
means = np.asarray(beta.mean(dim=("chain", "draw")), dtype=float)
figure, axis = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(levels))
axis.hlines(y, hdi[:, 0], hdi[:, 1], color="steelblue", lw=2.2)
axis.scatter(means, y, color="steelblue", s=22)
axis.axvline(0, color="black", lw=0.8)
axis.axvline(ATTENUATION_20, color="crimson", ls="--", lw=1)
axis.set_yticks(y)
axis.set_yticklabels(levels)
axis.invert_yaxis()
axis.set_xlabel("configuration log recovery contrast")
axis.set_title("Model A configuration effects with 95% intervals")
figure.tight_layout()
figure.savefig(FIGURE_ROOT / "A_configuration_forest.png", dpi=140, bbox_inches="tight")
plt.show()


## 4. Required Model A validation gates

### 4.1 Synthetic parameter recovery

This reduced synthetic fit starts from known effects and verifies that the model recovers them.
It is method validation only, never empirical H1 evidence.


In [ ]:
RECOVERY_CHECKPOINT = "03_A_recovery_v1.nc"
RECOVERY_TABLE = "03_A_recovery_v1.parquet"
RECOVERY_DRAWS, RECOVERY_TUNE = 2500, 1500


def sample_model(model, label, draws=DRAWS, tune=TUNE, target_accept=TARGET_ACCEPT):
    kwargs = {}
    if NUTS_BACKEND != "pymc":
        kwargs["nuts_sampler"] = NUTS_BACKEND
    print(f"sampling {label}: draws={draws}, tune={tune}, chains={CHAINS}, cores={CORES}")
    with model:
        return pm.sample(
            draws=draws, tune=tune, chains=CHAINS, cores=CORES,
            random_seed=SEED, target_accept=target_accept, progressbar=True,
            idata_kwargs={"log_likelihood": False}, **kwargs,
        )


def zero_sum(values):
    return values - values.mean()


def simulate_A_recovery(df, seed=SEED):
    rng = np.random.default_rng(seed)
    out = df[df["bg_id"] < 3].copy().reset_index(drop=True)
    config_code, config_levels = codes(out["model"])
    harmonic_code, harmonic_levels = codes(out["f_lock"].round(3).astype(str))
    background_code, background_levels = codes(
        out["generator"].astype(str) + "#" + out["bg_id"].astype(str)
    )
    truths = {"beta_bar": -0.40, "delta_O": 0.25, "delta_P": -0.15}
    mu = (
        truths["beta_bar"]
        + truths["delta_O"] * overlap_scaled(out, config_levels)[config_code]
        + truths["delta_P"] * log_patch_centred(out, config_levels)[config_code]
        + zero_sum(rng.normal(0, 0.10, len(config_levels)))[config_code]
        + zero_sum(rng.normal(0, 0.12, len(harmonic_levels)))[harmonic_code]
        + zero_sum(rng.normal(0, 0.08, len(background_levels)))[background_code]
    )
    out["d"] = mu + 0.25 * stats.t.rvs(
        NU, size=len(out), random_state=rng.integers(1 << 31)
    )
    return out, truths


if have_artifact(RECOVERY_TABLE):
    recovery_table = load_frame(RECOVERY_TABLE)
elif RUN_PARAMETER_RECOVERY:
    simulated_A, recovery_truths = simulate_A_recovery(live)
    if have_artifact(RECOVERY_CHECKPOINT):
        recovery_idata = load_idata(artifact_path(RECOVERY_CHECKPOINT))
    else:
        recovery_idata = sample_model(
            model_A(simulated_A), "A synthetic recovery",
            draws=RECOVERY_DRAWS, tune=RECOVERY_TUNE, target_accept=0.99,
        )
        save_idata(recovery_idata, RECOVERY_CHECKPOINT)
    recovery_diagnostic = bc.fit_diagnostics("recovery:A", recovery_idata, az)
    rows = []
    for parameter, truth in recovery_truths.items():
        values = np.asarray(recovery_idata.posterior[parameter], dtype=float).ravel()
        low, high = np.quantile(values, [0.025, 0.975])
        rows.append({
            "model": "A", "parameter": parameter, "truth": truth,
            "median": float(np.median(values)),
            "hdi_low": float(low), "hdi_high": float(high),
            "covered": bool(low <= truth <= high), **recovery_diagnostic,
        })
    recovery_table = save_frame(pd.DataFrame(rows), RECOVERY_TABLE)
    del simulated_A, recovery_idata
    gc.collect()
else:
    recovery_table = pd.DataFrame()

RECOVERY_OK = bool(
    not recovery_table.empty
    and recovery_table["covered"].astype(bool).all()
    and recovery_table["diagnostics_ok"].astype(bool).all()
)
display(recovery_table.round(5))
print("parameter recovery gate:", "PASS" if RECOVERY_OK else "FAIL OR NOT RUN")


### 4.2 Posterior-predictive check in draw blocks

Observed configuration and generator means are compared with 95 percent replicated intervals.
At least 90 percent of the required strata must pass. Every completed block is saved.


In [ ]:
PPC_TABLE_NAME = "05_A_ppc_v1.parquet"
PPC_PARTIAL_PATH = WORK_ROOT / "A_ppc_draw_blocks.parquet"


def ppc_positions(frame):
    positions = {}
    for column in ("model", "generator"):
        for level, index in frame.groupby(column, observed=True).indices.items():
            positions[f"A:{column}={level}"] = np.asarray(index, dtype=int)
    return positions


if have_artifact(PPC_TABLE_NAME):
    ppc_table = load_frame(PPC_TABLE_NAME)
else:
    positions = ppc_positions(live)
    observed = live["d"].to_numpy(float)
    n_draw = int(idata_A.posterior.sizes["draw"])
    starts = list(range(0, n_draw, PPC_DRAW_CHUNK))
    partial = pd.read_parquet(PPC_PARTIAL_PATH) if PPC_PARTIAL_PATH.is_file() else pd.DataFrame()
    completed = set(map(int, partial["block_start"].unique())) if not partial.empty else set()
    pending_total = sum(start not in completed for start in starts)
    new_done = 0
    ppc_model = model_A(live)
    started_at = time.time()
    for block_number, start in enumerate(starts, start=1):
        stop = min(start + PPC_DRAW_CHUNK, n_draw)
        if start in completed:
            print(f"PPC [{block_number}/{len(starts)}] draws {start}:{stop} cached")
            continue
        posterior_block = idata_A.isel(draw=slice(start, stop))
        with ppc_model:
            predictive = pm.sample_posterior_predictive(
                posterior_block, random_seed=SEED + start, progressbar=False
            )
        replicated = np.asarray(
            predictive.posterior_predictive["d"], dtype=float
        ).reshape(-1, len(live))
        block_frame = pd.DataFrame({
            stratum: replicated[:, index].mean(axis=1)
            for stratum, index in positions.items()
        })
        block_frame.insert(0, "block_start", start)
        partial = pd.concat([partial, block_frame], ignore_index=True)
        cp.atomic_parquet(PPC_PARTIAL_PATH, partial)
        new_done += 1
        elapsed = time.time() - started_at
        eta = elapsed / new_done * max(0, pending_total - new_done)
        print(f"PPC [{block_number}/{len(starts)}] draws {start}:{stop} saved; "
              f"elapsed {elapsed/60:.1f} min; ETA {eta/60:.1f} min")
        del posterior_block, predictive, replicated, block_frame
        gc.collect()
    if set(map(int, partial["block_start"].unique())) != set(starts):
        raise RuntimeError("PPC partial checkpoint is incomplete")
    rows = []
    for stratum, index in positions.items():
        values = partial[stratum].to_numpy(float)
        low, high = np.quantile(values, [0.025, 0.975])
        observed_mean = float(observed[index].mean())
        rows.append({
            "analysis": "A", "stratum": stratum, "n": len(index),
            "observed": observed_mean, "rep_low": float(low), "rep_high": float(high),
            "ppc_ok": bool(low <= observed_mean <= high),
        })
    ppc_table = save_frame(pd.DataFrame(rows), PPC_TABLE_NAME)
    del partial, ppc_model
    gc.collect()

required_ppc = set(ppc_positions(live))
PPC_OK = bc.ppc_gate(ppc_table, required_ppc, minimum_coverage=PPC_MIN_COVERAGE)
display(ppc_table)
print("PPC coverage:", f"{ppc_table['ppc_ok'].mean():.1%}")
print("posterior-predictive gate:", "PASS" if PPC_OK else "FAIL")

plot_table = ppc_table.sort_values("stratum").reset_index(drop=True)
figure, axis = plt.subplots(figsize=(10, max(4, 0.28 * len(plot_table))))
y = np.arange(len(plot_table))
axis.hlines(y, plot_table["rep_low"], plot_table["rep_high"],
            color="steelblue", lw=3, alpha=0.65)
axis.scatter(plot_table["observed"], y,
             c=np.where(plot_table["ppc_ok"], "black", "crimson"), s=18)
axis.set_yticks(y)
axis.set_yticklabels(plot_table["stratum"], fontsize=7)
axis.set_title("Model A PPC stratum means; red means outside the 95% interval")
figure.tight_layout()
figure.savefig(FIGURE_ROOT / "A_ppc.png", dpi=140, bbox_inches="tight")
plt.show()


## 5. Fit the three missing M1 variants

The base fit is the overlap plus patch-size member. The next three fits change only which of those
two predictors enters the configuration mean. Every checkpoint excludes log likelihood because
that large array is reconstructed later in blocks.


In [ ]:
def fit_or_load(level, label, filename):
    if have_artifact(filename):
        print("loading completed fit:", filename)
        return load_idata(artifact_path(filename))
    fitted = sample_model(model_A(live, config_level=level), f"A:{label}")
    save_idata(fitted, filename)
    return fitted


print("The next three cells are independently resumable.")


### 5.1 Fit or resume overlap only

The live progress table gives the real Colab speed and accounts for the two queued chains.


In [ ]:
idata_overlap = fit_or_load("overlap", "overlap only", "04_A_overlap_v2.nc")
print(bc.fit_diagnostics("A:overlap only", idata_overlap, az))


### 5.2 Fit or resume patch size only

The live progress table gives the real Colab speed and accounts for the two queued chains.


In [ ]:
idata_patch = fit_or_load("patch", "patch size only", "04_A_patch_v2.nc")
print(bc.fit_diagnostics("A:patch size only", idata_patch, az))


### 5.3 Fit or resume neither

The live progress table gives the real Colab speed and accounts for the two queued chains.


In [ ]:
idata_none = fit_or_load("none", "neither", "04_A_none_v2.nc")
print(bc.fit_diagnostics("A:neither", idata_none, az))


### 5.4 Diagnose all four M1 fits

The global project gate requires R-hat below 1.01, bulk and tail ESS above 1,000, and no
divergences. The exact M1 decision uses the base and patch-only pair; all four are also reported.


In [ ]:
FIT_IDATA = {
    "A": idata_A,
    "A:overlap-only": idata_overlap,
    "A:patch-only": idata_patch,
    "A:neither": idata_none,
}
diagnostics_table = bc.diagnostics_table(FIT_IDATA, az)
save_frame(diagnostics_table, "05_A_M1_diagnostics_v1.parquet")
display(diagnostics_table.round(5))
M1_DIAGNOSTICS_ALL_OK = bool(diagnostics_table["diagnostics_ok"].all())
by_fit = diagnostics_table.set_index("fit")
M1_DECISIVE_DIAGNOSTICS_OK = bool(
    by_fit.loc[["A", "A:patch-only"], "diagnostics_ok"].all()
)
print("all four fits:", "PASS" if M1_DIAGNOSTICS_ALL_OK else "FAIL")
print("decisive base versus patch pair:",
      "PASS" if M1_DECISIVE_DIAGNOSTICS_OK else "FAIL")


## 6. Prior and likelihood sensitivity

The current project gate requires H1 attenuation and M1 overlap probabilities to vary by no more
than 0.10 across reasonable alternatives. The primary Student-t scale 0.5 row reuses the completed
base fit. Only the remaining three alternatives are sampled.


In [ ]:
SENSITIVITY_SPECS = {
    "StudentT scale 0.25": (0.25, "student", "05_A_sensitivity_scale025_v1.nc"),
    "StudentT scale 1.0": (1.0, "student", "05_A_sensitivity_scale100_v1.nc"),
    "Normal likelihood scale 0.5": (0.5, "normal", "05_A_sensitivity_normal_v1.nc"),
}


def sensitivity_fit_or_load(label):
    scale, likelihood, filename = SENSITIVITY_SPECS[label]
    if have_artifact(filename):
        return load_idata(artifact_path(filename))
    if not RUN_SENSITIVITY:
        print("sensitivity disabled; missing:", label)
        return None
    fitted = sample_model(
        model_A(live, scale=scale, likelihood=likelihood, config_level="both"),
        f"sensitivity:{label}",
    )
    save_idata(fitted, filename)
    return fitted


### 6.1 Fit or resume StudentT scale 0.25


In [ ]:
sens_scale025 = sensitivity_fit_or_load("StudentT scale 0.25")


### 6.2 Fit or resume StudentT scale 1.0


In [ ]:
sens_scale100 = sensitivity_fit_or_load("StudentT scale 1.0")


### 6.3 Fit or resume Normal likelihood scale 0.5


In [ ]:
sens_normal = sensitivity_fit_or_load("Normal likelihood scale 0.5")


### 6.4 Evaluate the sensitivity gate


In [ ]:
def sensitivity_row(label, idata):
    diagnostic = bc.fit_diagnostics(f"sensitivity:{label}", idata, az)
    beta_values = np.asarray(idata.posterior["beta_bar"], dtype=float).ravel()
    delta_values = np.asarray(idata.posterior["delta_O"], dtype=float).ravel()
    low, high = np.quantile(beta_values, [0.025, 0.975])
    return {
        "variant": label,
        "median_beta": float(np.median(beta_values)),
        "beta_low": float(low), "beta_high": float(high),
        "P_attenuation_20": float(np.mean(beta_values < ATTENUATION_20)),
        "P_beta_in_rope": float(np.mean(np.abs(beta_values) < ROPE_LOG)),
        "P_delta_O_positive": float(np.mean(delta_values > 0)),
        **diagnostic,
    }


sensitivity_pairs = [("StudentT scale 0.5 primary reused", idata_A)]
for label, fitted in (
    ("StudentT scale 0.25", sens_scale025),
    ("StudentT scale 1.0", sens_scale100),
    ("Normal likelihood scale 0.5", sens_normal),
):
    if fitted is not None:
        sensitivity_pairs.append((label, fitted))

sensitivity_table = pd.DataFrame([
    sensitivity_row(label, fitted) for label, fitted in sensitivity_pairs
])
save_frame(sensitivity_table, "05_A_sensitivity_v1.parquet")
SENSITIVITY_OK = bool(
    len(sensitivity_table) == 4
    and sensitivity_table["diagnostics_ok"].astype(bool).all()
    and bc.sensitivity_gate(
        sensitivity_table,
        ("P_attenuation_20", "P_delta_O_positive"),
        max_spread=SENSITIVITY_MAX_SPREAD,
    )
)
display(sensitivity_table.round(5))
spreads = {
    column: float(sensitivity_table[column].max() - sensitivity_table[column].min())
    for column in ("P_attenuation_20", "P_delta_O_positive")
}
print("probability spreads:", spreads)
print("sensitivity gate:", "PASS" if SENSITIVITY_OK else "FAIL OR INCOMPLETE")
del sens_scale025, sens_scale100, sens_normal
gc.collect()


## 7. Memory-bounded analytic PSIS-LOO

The fit checkpoints intentionally contain no giant observation-wise log-likelihood array. This
section reconstructs the exact Student-t log likelihood from each saved posterior. Every block
contains all 8,000 posterior samples but only enough observations to target roughly 128 MiB for
its main matrix. Blocking changes memory use and resumability, not the scientific result.


In [ ]:
def gpdfit(ary):
    prior_bs, prior_k = 3, 10
    n = len(ary)
    m_est = 30 + int(n**0.5)
    b_ary = 1 - np.sqrt(m_est / (np.arange(1, m_est + 1, dtype=float) - 0.5))
    b_ary /= prior_bs * ary[int(n / 4 + 0.5) - 1]
    b_ary += 1 / ary[-1]
    k_ary = np.log1p(-b_ary[:, None] * ary).mean(axis=1)
    len_scale = n * (np.log(-(b_ary / k_ary)) - k_ary - 1)
    weights = 1 / np.exp(len_scale - len_scale[:, None]).sum(axis=1)
    real = weights >= 10 * np.finfo(float).eps
    weights, b_ary = weights[real], b_ary[real]
    weights /= weights.sum()
    b_post = np.sum(b_ary * weights)
    k_post = np.log1p(-b_post * ary).mean()
    sigma = -k_post / b_post
    k_post = (n * k_post + prior_k * 0.5) / (n + prior_k)
    return k_post, sigma


def gpinv(probs, kappa, sigma):
    x = np.full_like(probs, np.nan)
    if sigma <= 0:
        return x
    ok = (probs > 0) & (probs < 1)
    if np.abs(kappa) < np.finfo(float).eps:
        x[ok] = -np.log1p(-probs[ok])
    else:
        x[ok] = np.expm1(-kappa * np.log1p(-probs[ok])) / kappa
    x *= sigma
    x[probs == 0] = 0
    x[probs == 1] = np.inf if kappa >= 0 else -sigma / kappa
    return x


def psislw_1d(log_weights, cutoff_ind, cutoffmin):
    x = np.asarray(log_weights, dtype=float)
    x -= np.max(x)
    order = np.argsort(x)
    cutoff = max(x[order[cutoff_ind]], cutoffmin)
    exp_cutoff = np.exp(cutoff)
    tail_indices = np.where(x > cutoff)[0]
    tail = x[tail_indices]
    if len(tail) <= 4:
        k = np.inf
    else:
        tail_order = np.argsort(tail)
        tail = np.exp(tail) - exp_cutoff
        k, sigma = gpdfit(tail[tail_order])
        if np.isfinite(k):
            probabilities = np.arange(0.5, len(tail)) / len(tail)
            x[tail_indices[tail_order]] = np.log(
                gpinv(probabilities, k, sigma) + exp_cutoff
            )
            x[x > 0] = 0
    x -= logsumexp(x)
    return x, k


def psis_block(log_likelihood, cutoff_ind, cutoffmin):
    n_obs, n_samples = log_likelihood.shape
    log_n = np.log(n_samples)
    elpd_i = np.empty(n_obs)
    pareto_k = np.empty(n_obs)
    lppd_i = np.empty(n_obs)
    for index in range(n_obs):
        ll = log_likelihood[index]
        log_weights, k = psislw_1d(-ll, cutoff_ind, cutoffmin)
        elpd_i[index] = logsumexp(log_weights + ll)
        pareto_k[index] = k
        lppd_i[index] = logsumexp(ll) - log_n
    return elpd_i, pareto_k, lppd_i


STUDENT_T_LOG_CONSTANT = float(
    gammaln((NU + 1) / 2) - gammaln(NU / 2) - 0.5 * np.log(NU * np.pi)
)


def student_t_logpdf(y, mu, sigma):
    standardized = (y - mu) / sigma
    return (
        STUDENT_T_LOG_CONSTANT - np.log(sigma)
        - ((NU + 1) / 2) * np.log1p((standardized * standardized) / NU)
    )


test_y = np.array([-1.2, 0.0, 0.8])
test_mu = np.array([-0.9, 0.2, 1.1])
test_sigma = np.array([0.5, 1.2, 0.7])
reference = stats.t.logpdf(test_y, df=NU, loc=test_mu, scale=test_sigma)
if not np.allclose(
    student_t_logpdf(test_y, test_mu, test_sigma),
    reference, rtol=1e-12, atol=1e-12,
):
    raise AssertionError("analytic Student-t log likelihood differs from scipy")
print("analytic Student-t log likelihood: PASS against scipy")

# Guard the internal, version-stable PSIS port against ArviZ's public implementation.
if hasattr(az, "psislw"):
    test_rng = np.random.default_rng(SEED)
    test_ll = test_rng.normal(-1.0, 0.7, size=(3, 800))
    test_n = test_ll.shape[-1]
    test_cutoff = -int(np.ceil(min(test_n / 5.0, 3 * test_n**0.5))) - 1
    ours_elpd, ours_k, _ = psis_block(
        test_ll, test_cutoff, np.log(np.finfo(float).tiny)
    )
    reference_lw, reference_k = az.psislw(-test_ll, reff=1.0)
    reference_elpd = logsumexp(reference_lw + test_ll, axis=1)
    if not (
        np.allclose(ours_elpd, reference_elpd, rtol=1e-11, atol=1e-11)
        and np.allclose(ours_k, reference_k, rtol=1e-11, atol=1e-11)
    ):
        raise AssertionError("internal PSIS port differs from az.psislw")
    print("PSIS port: PASS against az.psislw")
else:
    print("PSIS public cross-check unavailable on this ArviZ version")


In [ ]:
def relative_efficiency(idata):
    posterior = idata.posterior
    n_chains = int(posterior.sizes["chain"])
    n_samples = n_chains * int(posterior.sizes["draw"])
    if n_chains == 1:
        return 1.0
    ess = az.ess(posterior, method="mean")
    values = np.hstack([np.asarray(ess[name]).ravel() for name in ess.data_vars])
    return float(np.mean(values) / n_samples)


def codes_against(values, levels, label):
    mapping = {str(level): index for index, level in enumerate(levels)}
    result = np.asarray([mapping.get(str(value), -1) for value in values], dtype=int)
    if np.any(result < 0):
        unknown = sorted({str(values[index]) for index in np.where(result < 0)[0]})
        raise ValueError(f"{label} levels absent from posterior coordinates: {unknown[:5]}")
    return result


def stack_posterior(array, level_dimension=None):
    dimensions = ["chain", "draw"] + ([level_dimension] if level_dimension else [])
    values = np.asarray(array.transpose(*dimensions), dtype=np.float64)
    if level_dimension:
        return values.reshape(-1, values.shape[-1])
    return values.reshape(-1)


def posterior_likelihood_inputs(idata, frame):
    posterior = idata.posterior
    config_levels = list(map(str, posterior.coords["config"].values))
    harmonic_levels = list(map(str, posterior.coords["harmonic"].values))
    background_levels = list(map(str, posterior.coords["background"].values))
    config_code = codes_against(frame["model"].astype(str), config_levels, "config")
    harmonic_code = codes_against(
        frame["f_lock"].round(3).astype(str), harmonic_levels, "harmonic"
    )
    background_key = frame["generator"].astype(str) + "#" + frame["bg_id"].astype(str)
    background_code = codes_against(background_key, background_levels, "background")
    if "u_bg" in posterior:
        u_bg = stack_posterior(posterior["u_bg"], "background")
    else:
        u_bg = (
            stack_posterior(posterior["sigma_bg"])[:, None]
            * stack_posterior(posterior["z_bg"], "background")
        )
    return {
        "beta": stack_posterior(posterior["beta"], "config"),
        "u_harm": stack_posterior(posterior["u_harm"], "harmonic"),
        "u_bg": u_bg,
        "sigma": stack_posterior(posterior["sigma"]),
        "config_code": config_code,
        "harmonic_code": harmonic_code,
        "background_code": background_code,
        "y": frame["d"].to_numpy(float),
    }


def loo_names(fit_key):
    return f"loo_pointwise__{fit_key}.parquet", f"loo_scalars__{fit_key}.json"


def analytic_loo_cached(fit_key, checkpoint):
    pointwise_name, scalar_name = loo_names(fit_key)
    if have_artifact(pointwise_name) and have_artifact(scalar_name):
        pointwise = load_frame(pointwise_name)
        result = load_value(scalar_name)
        result["elpd_i"] = pointwise["elpd_i"].to_numpy(float)
        result["pareto_k"] = pointwise["pareto_k"].to_numpy(float)
        print("LOO reused:", fit_key)
        return result

    idata = load_idata(checkpoint)
    inputs = posterior_likelihood_inputs(idata, live)
    n_samples = len(inputs["sigma"])
    n_obs = len(live)
    reff = relative_efficiency(idata)
    cutoff_ind = -int(np.ceil(
        min(n_samples / 5.0, 3 * (n_samples / reff) ** 0.5)
    )) - 1
    cutoffmin = np.log(np.finfo(float).tiny)
    per_obs_bytes = n_samples * 8
    chunk_size = max(1, min(n_obs, int(LOO_TARGET_BYTES // per_obs_bytes)))
    starts = list(range(0, n_obs, chunk_size))
    work_directory = WORK_ROOT / f"loo_{fit_key}"
    work_directory.mkdir(parents=True, exist_ok=True)
    expected_chunks = [
        work_directory / f"chunk_{start:06d}_{min(start + chunk_size, n_obs):06d}.npz"
        for start in starts
    ]
    pending_total = sum(not path.is_file() for path in expected_chunks)
    new_done = 0
    print(f"LOO {fit_key}: {n_obs:,} observations x {n_samples:,} samples; "
          f"{len(starts)} blocks of at most {chunk_size:,} observations")

    started_at = time.time()
    all_elpd, all_k, all_lppd = [], [], []
    for block_number, start in enumerate(starts, start=1):
        stop = min(start + chunk_size, n_obs)
        chunk_path = work_directory / f"chunk_{start:06d}_{stop:06d}.npz"
        if chunk_path.is_file():
            with np.load(chunk_path) as cached:
                elpd_i = np.asarray(cached["elpd_i"], dtype=float)
                pareto_k = np.asarray(cached["pareto_k"], dtype=float)
                lppd_i = np.asarray(cached["lppd_i"], dtype=float)
            if not (len(elpd_i) == len(pareto_k) == len(lppd_i) == stop - start):
                raise ValueError(f"invalid LOO block: {chunk_path}")
            state = "cached"
        else:
            config = inputs["config_code"][start:stop]
            harmonic = inputs["harmonic_code"][start:stop]
            background = inputs["background_code"][start:stop]
            location = (
                inputs["beta"][:, config]
                + inputs["u_harm"][:, harmonic]
                + inputs["u_bg"][:, background]
            )
            sigma = inputs["sigma"][:, None]
            observed = inputs["y"][None, start:stop]
            log_likelihood = student_t_logpdf(observed, location, sigma).T
            elpd_i, pareto_k, lppd_i = psis_block(
                log_likelihood, cutoff_ind, cutoffmin
            )
            temporary = chunk_path.with_name(chunk_path.stem + ".tmp.npz")
            np.savez_compressed(
                temporary, elpd_i=elpd_i, pareto_k=pareto_k, lppd_i=lppd_i
            )
            os.replace(temporary, chunk_path)
            del location, sigma, observed, log_likelihood
            gc.collect()
            state = "saved"
            new_done += 1
        all_elpd.append(elpd_i)
        all_k.append(pareto_k)
        all_lppd.append(lppd_i)
        elapsed = time.time() - started_at
        eta = (
            elapsed / new_done * max(0, pending_total - new_done)
            if new_done else 0.0
        )
        print(f"LOO {fit_key} [{block_number}/{len(starts)}] obs {start}:{stop} {state}; "
              f"elapsed {elapsed/60:.1f} min; ETA {eta/60:.1f} min")

    elpd_i = np.concatenate(all_elpd)
    pareto_k = np.concatenate(all_k)
    lppd = float(np.concatenate(all_lppd).sum())
    elpd_loo = float(elpd_i.sum())
    se = float(np.sqrt(n_obs * np.var(elpd_i)))
    p_loo = float(lppd - elpd_loo)
    good_k = float(min(1 - 1 / np.log10(n_samples), 0.7))
    warning = bool(np.any(pareto_k > good_k))
    result = {
        "fit": fit_key, "elpd_loo": elpd_loo, "se": se, "p_loo": p_loo,
        "n_samples": n_samples, "n_obs": n_obs, "reff": reff,
        "warning": warning, "good_k": good_k,
        "max_pareto_k": float(np.nanmax(pareto_k)),
    }
    save_frame(
        pd.DataFrame({"elpd_i": elpd_i, "pareto_k": pareto_k}),
        pointwise_name,
    )
    save_value(result, scalar_name)
    result["elpd_i"] = elpd_i
    result["pareto_k"] = pareto_k
    del idata, inputs, all_elpd, all_k, all_lppd
    gc.collect()
    return result


### 7.1 Calculate or resume four pointwise LOO results

The cell prints block progress, elapsed time and ETA. Completed blocks are reused.


In [ ]:
FIT_PATHS = {
    "overlap + patch size": BASE_CHECKPOINT,
    "overlap only": artifact_path("04_A_overlap_v2.nc"),
    "patch size only": artifact_path("04_A_patch_v2.nc"),
    "neither": artifact_path("04_A_none_v2.nc"),
}
fit_keys = {
    "overlap + patch size": "A_both",
    "overlap only": "A_overlap",
    "patch size only": "A_patch",
    "neither": "A_none",
}
loo_streams = {}
for label, checkpoint in FIT_PATHS.items():
    loo_streams[label] = analytic_loo_cached(fit_keys[label], checkpoint)
    gc.collect()
print("all four pointwise LOO results are available")


### 7.2 Compare M1 variants and check Pareto-k reliability

Expected log predictive density measures prediction, dse measures uncertainty in a difference,
and Pareto-k checks whether the efficient LOO approximation is reliable. The registered M1
decision compares overlap plus patch size directly with patch size only.


In [ ]:
def elpd_data(stream):
    elpd_i = xr.DataArray(stream["elpd_i"], dims=["__obs__"])
    pareto_k = xr.DataArray(stream["pareto_k"], dims=["__obs__"])
    try:
        from arviz_stats.utils import ELPDData as ELPDData1
        return ELPDData1(
            "loo", stream["elpd_loo"], stream["se"], stream["p_loo"],
            stream["n_samples"], stream["n_obs"], "log", stream["warning"],
            stream["good_k"], elpd_i, pareto_k,
            approx_posterior=False, log_weights=None,
        )
    except Exception:
        from arviz.stats import ELPDData as ELPDData0
        return ELPDData0(
            data=[
                stream["elpd_loo"], stream["se"], stream["p_loo"],
                stream["n_samples"], stream["n_obs"], stream["warning"],
                elpd_i.rename("loo_i"), pareto_k, "log", stream["good_k"],
            ],
            index=[
                "elpd_loo", "se", "p_loo", "n_samples", "n_data_points",
                "warning", "loo_i", "pareto_k", "scale", "good_k",
            ],
        )


def manual_compare(streams):
    names = sorted(streams, key=lambda name: -streams[name]["elpd_loo"])
    best = names[0]
    rows = []
    for rank, name in enumerate(names):
        difference = streams[best]["elpd_i"] - streams[name]["elpd_i"]
        rows.append({
            "model": name, "rank": rank,
            "elpd_loo": streams[name]["elpd_loo"],
            "p_loo": streams[name]["p_loo"],
            "elpd_diff": float(difference.sum()), "weight": np.nan,
            "se": streams[name]["se"],
            "dse": float(np.sqrt(len(difference) * np.var(difference))),
            "warning": streams[name]["warning"], "scale": "log",
        })
    return pd.DataFrame(rows).set_index("model")


def compare_streams(streams):
    try:
        objects = {name: elpd_data(value) for name, value in streams.items()}
        if "ic" in inspect.signature(az.compare).parameters:
            comparison = az.compare(objects, ic="loo")
        else:
            comparison = az.compare(objects)
        if {"elpd_diff", "dse"}.issubset(comparison.columns):
            return comparison
    except Exception as error:
        print("az.compare fallback:", repr(error))
    return manual_compare(streams)


def quality_table(streams):
    rows = []
    for name, stream in streams.items():
        reliable = bool(
            np.isfinite(stream["max_pareto_k"])
            and not stream["warning"]
            and stream["max_pareto_k"] <= stream["good_k"]
        )
        rows.append({
            "fit": name, "max_pareto_k": stream["max_pareto_k"],
            "good_k": stream["good_k"], "loo_warning": stream["warning"],
            "loo_reliable": reliable,
        })
    return pd.DataFrame(rows).set_index("fit")


def comparison_decision(comparison, quality):
    winner, runner_up = str(comparison.index[0]), str(comparison.index[1])
    margin = float(comparison.iloc[1]["elpd_diff"])
    dse = float(comparison.iloc[1]["dse"])
    reliable = bool(quality["loo_reliable"].all())
    separated = bool(np.isfinite([margin, dse]).all() and margin >= 2 * dse)
    return {
        "winner": winner, "runner_up": runner_up,
        "elpd_margin": margin, "dse": dse,
        "loo_reliable": reliable, "separated_2dse": separated,
        "comparison_ok": bool(reliable and separated),
    }


comparison_all = compare_streams(loo_streams)
quality_all = quality_table(loo_streams)
decision_all = comparison_decision(comparison_all, quality_all)
exact_names = ("overlap + patch size", "patch size only")
exact_streams = {name: loo_streams[name] for name in exact_names}
comparison_exact = compare_streams(exact_streams)
quality_exact = quality_table(exact_streams)
decision_exact = comparison_decision(comparison_exact, quality_exact)

save_frame(
    comparison_all.rename_axis("model").reset_index(),
    "05_M1_loo_fourway_comparison_v1.parquet",
)
save_frame(
    quality_all.rename_axis("fit").reset_index(),
    "05_M1_loo_fourway_quality_v1.parquet",
)
save_value(decision_all, "05_M1_loo_fourway_decision_v1.json")
save_frame(
    comparison_exact.rename_axis("model").reset_index(),
    "05_M1_loo_exact_comparison_v1.parquet",
)
save_frame(
    quality_exact.rename_axis("fit").reset_index(),
    "05_M1_loo_exact_quality_v1.parquet",
)
save_value(decision_exact, "05_M1_loo_exact_decision_v1.json")

display(comparison_all)
display(quality_all)
print("four-way:", decision_all)
print("exact full versus patch-only comparison")
display(comparison_exact)
display(quality_exact)
print(decision_exact)


## 8. Final fail-closed H1 and M1 verdicts

A failed or missing validation gate yields NOT REPORTABLE and is never converted into a false
hypothesis. With all gates passed, the registered rules return TRUE, FALSE or INCONCLUSIVE.


In [ ]:
H1_GATE_OK = bool(
    base_diagnostic["diagnostics_ok"] and RECOVERY_OK and PPC_OK and SENSITIVITY_OK
)
h1_project_verdict = bc.three_way_verdict(
    P_ATTENUATION_20, P_BETA_ROPE, H1_GATE_OK
)

M1_LOO_WIN = bc.required_loo_win(decision_exact, "overlap + patch size")
M1_GATE_OK = bool(
    M1_DECISIVE_DIAGNOSTICS_OK
    and RECOVERY_OK
    and PPC_OK
    and SENSITIVITY_OK
    and decision_exact["loo_reliable"]
)
if not M1_GATE_OK:
    m1_project_verdict = "NOT REPORTABLE"
elif P_DELTA_O_POSITIVE >= bc.POSTERIOR_CUTOFF and M1_LOO_WIN:
    m1_project_verdict = "supported"
elif P_DELTA_O_POSITIVE <= 1 - bc.POSTERIOR_CUTOFF:
    m1_project_verdict = "refuted"
else:
    m1_project_verdict = "inconclusive"


def tfi_label(project_verdict):
    return {
        "supported": "TRUE",
        "refuted": "FALSE",
        "inconclusive": "INCONCLUSIVE",
        "NOT REPORTABLE": "NOT REPORTABLE",
    }[project_verdict]


gate_table = pd.DataFrame([
    {"claim": "H1", "gate": "base convergence",
     "passed": bool(base_diagnostic["diagnostics_ok"])},
    {"claim": "H1 and M1", "gate": "parameter recovery", "passed": RECOVERY_OK},
    {"claim": "H1 and M1", "gate": "PPC", "passed": PPC_OK},
    {"claim": "H1 and M1", "gate": "sensitivity", "passed": SENSITIVITY_OK},
    {"claim": "M1 descriptive", "gate": "all four fits converge",
     "passed": M1_DIAGNOSTICS_ALL_OK},
    {"claim": "M1 decisive", "gate": "base and patch-only converge",
     "passed": M1_DECISIVE_DIAGNOSTICS_OK},
    {"claim": "M1 decisive", "gate": "exact LOO reliable",
     "passed": bool(decision_exact["loo_reliable"])},
])

verdicts = pd.DataFrame([
    {
        "hypothesis": "H1 behavioural", "model": "A",
        "support_probability": P_ATTENUATION_20,
        "refute_probability": P_BETA_ROPE,
        "gate_ok": H1_GATE_OK,
        "project_verdict": h1_project_verdict,
        "TFI": tfi_label(h1_project_verdict),
        "rule": "P(beta_bar < log 0.8)>=0.95; refute by beta ROPE>=0.95",
    },
    {
        "hypothesis": "M1 overlap mitigation", "model": "A'",
        "support_probability": P_DELTA_O_POSITIVE,
        "refute_probability": 1 - P_DELTA_O_POSITIVE,
        "gate_ok": M1_GATE_OK,
        "project_verdict": m1_project_verdict,
        "TFI": tfi_label(m1_project_verdict),
        "rule": "P(delta_O>0)>=0.95 and full model wins reliable LOO by >=2*dse",
    },
])

save_frame(gate_table, "05_A_M1_gate_table_v1.parquet")
save_frame(verdicts, "05_A_M1_verdicts_v1.parquet")
verdict_records = json.loads(verdicts.to_json(orient="records"))
save_value({
    "H1_gate_ok": H1_GATE_OK,
    "M1_gate_ok": M1_GATE_OK,
    "M1_LOO_WIN": M1_LOO_WIN,
    "H1": verdict_records[0],
    "M1": verdict_records[1],
    "exact_loo": decision_exact,
}, "05_A_M1_final_v1.json")

display(gate_table)
display(verdicts)
print("FINAL")
for row in verdicts.itertuples(index=False):
    print(f"{row.hypothesis}: {row.TFI}")
print("all outputs:", OUTPUT_ROOT)


## 9. How to read the result

- TRUE for H1 means at least 20 percent attenuation is supported and every validation gate passes.
- FALSE for H1 means the effect is concentrated inside the project negligible-effect region.
- INCONCLUSIVE means the fit is reportable but neither probability reached 0.95.
- NOT REPORTABLE means at least one gate failed; inspect the gate table instead of the probability.
- TRUE for M1 requires both a positive overlap effect with probability at least 0.95 and a reliable
  LOO win of overlap plus patch size over patch size only by at least twice the uncertainty.

The four-way M1 table remains useful descriptively. The registered decision uses the exact
base-versus-patch-only comparison.
